Inspect the physical execution plan. Do not assume a join strategy: verify the plan produced by the current Databricks runtime.


In [0]:
from src.config import config
from src.transformation import enrich_with_franchises, build_daily_sales_franchise_metrics

#### Silver inputs and franchise enrichment

In [0]:
transactions_df = spark.table(config.silver_transactions)
franchises_df = spark.table(config.silver_franchises)

enriched_df = enrich_with_franchises(
    transactions_df,
    franchises_df,
    cols_franchises=["franchiseID","franchiseName","city","district","country","size"],
)
enriched_df.explain("formatted")

#### Gold aggregation and ranking


In [0]:
gold_plan_df = build_daily_sales_franchise_metrics(enriched_df)
gold_plan_df.explain("formatted")

Inspect column pruning, join strategy, exchanges for aggregation/countDistinct, partitioning by `transaction_date`, and sorting for `dense_rank`.

In [0]:
gold_plan_df.orderBy("transaction_date","rank_sum_revenue").display()